# Schedule an energy demand forecast from a notebook

Press play to schedule the same data job on Pipekit as a recurring `CronWorkflow`: aggregate synthetic half-hourly demand into daily totals and forecast the next day, per region. No Kubernetes, no YAML, no `kubectl`.

The platform team owns `dataplatform/` (image, resources, service account, namespace, connection). You write a single job and schedule it. The one-off version of this job is in `run_forecast.ipynb`.

## Prerequisites

Install the Pipekit CLI, then log in once. This writes a token to `~/.pipekit/token` that the SDK reuses, so you do not paste a token into the notebook.

```bash
brew install pipekit/tap/cli   # or see https://docs.pipekit.io/cli
pipekit login
```

This repo ships a Nix dev shell (`shell.nix` at the repo root) that provides Python and the C++ runtime the Jupyter kernel needs on NixOS. From the repo root, inside the dev shell:

```bash
direnv allow                 # or run: nix-shell
python -m venv .venv
.venv/bin/pip install hera pipekit-sdk ipykernel jupyter
```

On NixOS the kernel needs `libstdc++` on `LD_LIBRARY_PATH`, or `pyzmq` fails to load and the editor reports that `jupyter and notebook` are missing. Bake that path into the kernel so it works however you launch your editor:

```bash
.venv/bin/python -m ipykernel install --user --name hera-forecast \
  --display-name "Python (hera-forecast)" \
  --env LD_LIBRARY_PATH "$LD_LIBRARY_PATH"
```

Then select the `Python (hera-forecast)` kernel for this notebook (kernel picker, top right), not a bare `Python 3.x` interpreter. Re-run the install command after a nixpkgs upgrade, since the baked path points into the Nix store.

pandas and numpy run on the cluster, not locally, so you do not need them here.

## Setup

In [ ]:
import os
import sys

# VSCode starts the kernel at the workspace root, not this folder. Point the
# working dir at this example so `dataplatform` and `forecast_step` import.
nb = globals().get('__vsc_ipynb_file__')
here = os.path.dirname(nb) if nb else os.getcwd()
if not os.path.isdir(os.path.join(here, 'dataplatform')):
    here = os.path.join(os.getcwd(), 'examples', 'hera-notebook-forecast')
os.chdir(here)
sys.path.insert(0, here)
print('working dir:', os.getcwd())

## Point at your cluster

Set the name of your free trial cluster. If you used a different name when you provisioned it, change it here. The SDK targets `https://api.pipekit.io` by default, which is correct for the free trial; set `PIPEKIT_URL` only for a different environment.

In [ ]:
os.environ['PIPEKIT_CLUSTER'] = 'free-trial-cluster'

## What you schedule

The same single function as the one-off run, now on a schedule. `forecast_demand` (in `forecast_step.py`) is a Hera `@script` step that aggregates demand and forecasts the next day with pandas. The platform image, resources, service account, and namespace come from `dataplatform`; you never set them.

`cron` builds a `CronWorkflow` from that step and creates it on Pipekit. The schedule is a standard cron expression: `0 6 * * *` runs the job at 06:00 every day. The call is idempotent on the name: run it again with a changed schedule and it updates the existing cron, so there is no delete-first step. The lifecycle helpers below need pipekit-sdk 2.1.2 or newer.

In [ ]:
from dataplatform import cron
from forecast_step import forecast_demand

cron('daily-demand-forecast', forecast_demand, schedule='0 6 * * *')

## Watch the run history

Open the link printed above. The cron has no runs yet; each tick of the schedule starts a new run, and they collect on that page over time. To see a run now without waiting for the schedule, trigger the cron once from the CLI:

```bash
pipekit trigger cron --cluster-name=free-trial-cluster --namespace=argo daily-demand-forecast
```

The `--namespace=argo` flag matches the namespace the platform defaults use. Without it, the CLI looks in the wrong namespace and reports that the cron does not exist. Triggering an on-demand run is a CLI or UI action; the SDK does not have it. The cells below manage the rest of the cron's lifecycle (update, check, suspend, resume) from the SDK. See the [cron CLI commands](https://docs.pipekit.io/cli/cron-workflows) for the CLI equivalents.

## Update the schedule

`cron` is idempotent on the name. Run it again with a different schedule and it updates the existing cron in place, with no delete first. Here the forecast moves from once a day to every four hours.

In [ ]:
cron('daily-demand-forecast', forecast_demand, schedule='0 */4 * * *')

## Check the cron

`get_cron` fetches the live cron and prints its schedule and whether it is suspended. It returns the full cron spec, so you can read other fields off the returned object.

In [ ]:
from dataplatform import get_cron

get_cron('daily-demand-forecast')

## Suspend and resume

`suspend_cron` pauses the schedule so no new runs start; runs already going keep running. `resume_cron` turns it back on. The `get_cron` call in between shows the state flip from active to suspended.

In [ ]:
from dataplatform import resume_cron, suspend_cron

suspend_cron('daily-demand-forecast')
get_cron('daily-demand-forecast')
resume_cron('daily-demand-forecast')

## The same cron as YAML (the GitOps path)

`cron_to_yaml` renders the manifest you would commit to a feature branch, without creating the cron. Same Python, now a versioned artifact. It is the same kind of `CronWorkflow` manifest as the native `examples/cronworkflow-example/workflow.yaml`, so you can move between the SDK path and a checked-in manifest without rewriting the job. One difference: this manifest puts the schedule in the `schedules` list, which Argo Workflows 3.6 requires, while the older native example uses the deprecated singular `schedule` field.

In [ ]:
from dataplatform import cron_to_yaml

print(cron_to_yaml('daily-demand-forecast', forecast_demand, schedule='0 6 * * *'))